# TRI-MODEL WITH SHARED TAK1

In [ ]:
# display("image/png", read("triModelA20Diagram.png"))

In [ ]:
###########################################################################
### File locations  ######################################################
###########################################################################
### set up where CSV2Julia is
locationOfCSV2Julia="csv2Julia/csv2model-multiscale.py"
#we use the same initial conditions at the PNAS paper 2019 Mitchell Roy et al.
fullModelInitFile="initialConditions/fullModelInits.csv"
moduleDefinitionFilesFolder="moduleDefinitionFiles/"
locationOfFixSpexies="utilityFunctions/"

In [ ]:
#packages we need
using DifferentialEquations
using Plots 
using Pkg
# Pkg.add(Pkg.PackageSpec(;name="Parsers", version="2.2.4"))
using CSV
using DataFrames
using JLD2
using FileIO
using StatsPlots
using Plots.PlotMeasures
using Distributions
delay=true

In [ ]:
include("combineModels.jl")
combinedModelLocation=combineModels(["BCR","TLR","NFkB"])

In [ ]:
include("generateModel.jl")

In [ ]:
include("defineInputs.jl")

#bcrSignalSS
#bcrSignalSSHigh
#tlrSignalSS
#bcrSignalTC
#tlrSignalTC
#nikSignalSS
#nikSignalTC

t=0
include("distributedModelFiles/odeModel.jl")
include("variableNames.jl")
include("scanIncludes.jl")
# const CBSWITCH_IDX = findfirst(==("cbswitch"), syms)
const CpGIndex = findfirst(==("CpGmedia"),syms)

include("runSimulation.jl")
include("plotAllSpecies.jl")
colorArray=palette(:seaborn_colorblind)

first_cell=1
last_cell=15
preCV=0.11
maxTimeTC=24*60
maxTimeSS=1000*60

In [ ]:

σ = 1.8
μ = σ^2
timeOfStimDist=Truncated(LogNormal(μ, σ),0,720)
TLRTimeArray=ones(1,last_cell)

heightDist=Truncated(Normal(1, 1),-Inf,10)d
TLRHeightArray=ones(1,last_cell)

globalCellIndex=1
for i in 1:last_cell
    y=rand(timeOfStimDist,1)
    TLRTimeArray[i]=y[1]
    x=maximum([rand(heightDist,1)[1],0])
    TLRHeightArray[i]=x[1]
end

In [ ]:
conditions=[]
paramsToChange=[]
modifyAmount=[]
BCRSSArray=[]
BCRTCArray=[]
TLRSSArray=[]
TLRTCArray=[]
NIKSSArray=[]
NIKTCArray=[]


# The parameter changes below have a fold change of 1 = no change. They are just a placeholder.
#M-CLL
push!(conditions,"Low BCR + CpG")
push!(paramsToChange,["Kd1_p100Synth-NFkB"])
push!(modifyAmount,[1.0])
push!(BCRSSArray,bcrSignalSS)
push!(BCRTCArray,bcrSignalSS)
push!(TLRSSArray,tlrSignalSS)
push!(TLRTCArray,tlrSignalSmooth)
push!(NIKSSArray,nikSignalSS)
push!(NIKTCArray,nikSignalSS)

# #U-CLL
push!(conditions,"High BCR + CpG")
push!(paramsToChange,["Kd1_p100Synth-NFkB"])
push!(modifyAmount,[1.0])
push!(BCRSSArray,bcrSignalSSHigh)
push!(BCRTCArray,bcrSignalSSHigh)
push!(TLRSSArray,tlrSignalSS)
push!(TLRTCArray,tlrSignalSmooth)
push!(NIKSSArray,nikSignalSS)
push!(NIKTCArray,nikSignalSS)
# put back in when working


println("Summary of conditions being run:")
folder="teachingOutputsStrongDelta"
show(IOContext(stdout, :limit => false), "text/plain", hcat(conditions,paramsToChange,modifyAmount))

In [ ]:
include("runSimulation.jl")

runSimulation(first_cell,last_cell,conditions,folder,BCRSSArray,BCRTCArray,TLRSSArray,TLRTCArray,NIKSSArray,NIKTCArray)

In [ ]:
include("plotAllSpeciesNew.jl")
hoursToPlot=7
colorArray=palette(:seaborn_colorblind)
speciesToPlot=syms
maxValOfYAxis=500
speciesToPlot=["IkBa*","RelAnp50n"]
#let't plot!
qualityScaling=0.5
gr()
plotAllSpeciesNew(speciesToPlot,conditions,colorArray,first_cell,last_cell,folder,hoursToPlot,maxValOfYAxis,qualityScaling,60)
